# Radar Classificador Colab — teste seu adaptador salvo

**Versão em preparação: homologação completa em segunda conta pendente.**
Seu PC abre o navegador; treino e inferência usam a GPU do Colab.
Este exemplo classifica mensagens em categoria e urgência. Não é um chatbot.
Use somente mensagens fictícias. Nenhum peso ou Drive privado do autor é necessário.
O corpus é sintético, sem aprovação humana; não valida atendimento real.

Salve uma cópia no seu Drive. Execute uma célula por vez, na ordem.
Cada etapa tem sua própria caixa de ativação; todas começam desmarcadas.
Não use “Executar tudo”. Leia o [guia do projeto](https://github.com/ofelipepeixoto/radar-classificador-colab/blob/feat/community-initial/docs/COMECE-AQUI.md).


## Configuração — informe a mesma pasta e execução do seu treino

Altere somente os dois campos abaixo. Use letras, números, `_` ou `-` e `/` entre pastas.
Este notebook aceita apenas adaptadores com selo de integridade criado pelo novo projeto. O piloto antigo do autor continua no notebook original.

In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, json, re, shutil, subprocess, sys

PASTA_PROJETO = 'RadarClassificador/meu-piloto-001' # @param {type:"string"}
NOME_EXECUCAO = 'run-001' # @param {type:"string"}

REPO_DIR = Path('/content/radar-classificador-colab')
if (not PASTA_PROJETO or any(not re.fullmatch(r'[A-Za-z0-9_-]{1,64}', p)
                           for p in PASTA_PROJETO.split('/'))
        or len(PASTA_PROJETO) > 150
        or not re.fullmatch(r'[A-Za-z0-9_-]{1,64}', NOME_EXECUCAO)):
    raise ValueError('Use nomes simples e pasta relativa como RadarClassificador/meu-piloto-001.')
ROOT = Path('/content/drive/MyDrive') / PASTA_PROJETO
CORPUS = ROOT / 'synthetic-machine-checked.jsonl'
FROZEN = ROOT / 'frozen'
RUN_DIR = ROOT / NOME_EXECUCAO
print('Sua pasta:', ROOT)
print('Sua execução:', RUN_DIR.name)


## 1. Confira o ambiente

Selecione um runtime com GPU na interface do Colab e confira as unidades da sua conta.
Não é necessário escolher a GPU mais cara. A GPU e o tempo disponíveis variam.
O ambiente de origem usou Python 3.12; a nova cópia ainda precisa de teste GPU completo.

In [ ]:
print('Python:', sys.version.split()[0])
gpu = shutil.which('nvidia-smi')
if gpu:
    result = subprocess.run([gpu, '--query-gpu=name,memory.total', '--format=csv,noheader'],
                            capture_output=True, text=True, timeout=30)
    print('GPU:', result.stdout.strip() if result.returncode == 0 else result.stderr.strip())
else:
    print('GPU não detectada. Selecione GPU antes de treinar ou abrir a interface.')


## 2. Obtenha o código e execute os testes

Marque **OBTER_CODIGO** e execute esta célula. O conteúdo Python e os dados
são conferidos por SHA-256 antes de qualquer importação do projeto.
Uma cópia existente é validada; não há atualização ou sobrescrita automática.

In [ ]:
OBTER_CODIGO = False # @param {type:"boolean"}
if OBTER_CODIGO:
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'feat/community-initial',
                        'https://github.com/ofelipepeixoto/radar-classificador-colab.git', str(REPO_DIR)], check=True, timeout=180)
    if REPO_DIR.is_symlink():
        raise ValueError('Pasta do código não pode ser um link.')
    # Only source files belong to this package. Reject stale or unlisted binaries.
    if any(p.suffix in {'.pyc', '.pyo', '.so', '.pyd'}
           for folder in ('ft05', 'scripts') for p in (REPO_DIR / folder).rglob('*') if p.is_file()):
        raise ValueError('Código compilado inesperado. Use uma sessão limpa.')
    sys.dont_write_bytecode = True
    manifest_file = REPO_DIR / 'SOURCE-MANIFEST.json'
    if hashlib.sha256(manifest_file.read_bytes()).hexdigest() != '807f73966cadaa5730848f1ab0b73c9be99b1dad5aea8344dda909fc4fb6c89f':
        raise ValueError('Versão do código diferente. Abra uma cópia atual do notebook em sessão limpa.')
    manifest = json.loads(manifest_file.read_text())
    for relative, expected in manifest.items():
        path = REPO_DIR / relative
        if (not path.resolve().is_relative_to(REPO_DIR.resolve()) or path.is_symlink()
                or hashlib.sha256(path.read_bytes()).hexdigest() != expected):
            raise ValueError('Arquivo de código/dados alterado: ' + relative)
    # Reject extra importable modules; a checked file must not import an unlisted one.
    actual = {str(p.relative_to(REPO_DIR)) for folder in ('ft05', 'scripts')
               for p in (REPO_DIR / folder).rglob('*')
               if p.is_file() and '__pycache__' not in p.parts
               and p.suffix in {'.py', '.json', '.jsonl', '.csv'}}
    if actual != set(manifest):
        raise ValueError('Arquivos inesperados no código. Use uma sessão limpa.')
    sys.path.insert(0, str(REPO_DIR / 'ft05'))
    commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
                                     text=True, timeout=30).strip()
    print('Commit do código:', commit)
    subprocess.run([sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
                   cwd=REPO_DIR / 'ft05', check=True, timeout=120)
    print('Código e testes: OK. Nenhuma biblioteca GPU foi instalada.')
else:
    print('Marque OBTER_CODIGO e execute esta célula para continuar.')


## 3. Monte o seu Drive

Marque **MONTAR_DRIVE**.
Autorize somente sua conta. O código passa a ter acesso ao Drive montado.
A verificação inicial não carrega pesos e recusa arquivos alterados.

In [ ]:
MONTAR_DRIVE = False # @param {type:"boolean"}
if MONTAR_DRIVE:
    if not OBTER_CODIGO:
        raise ValueError('Conclua primeiro a etapa 2 nesta sessão.')
    from google.colab import drive
    drive.mount('/content/drive')
    from community import paths
    selected = paths(PASTA_PROJETO, NOME_EXECUCAO)
    ROOT, CORPUS, FROZEN, RUN_DIR = [selected[k] for k in ('root', 'corpus', 'frozen', 'run')]
    from personal_smoke import preflight
    checked = preflight(RUN_DIR, CORPUS, FROZEN, versions=False)
    print('Corpus, split e selo do adaptador conferidos. Modelo:', checked['meta']['model'])
else:
    print('Drive não montado por esta célula. Marque MONTAR_DRIVE para continuar.')


## 4. Confira as dependências

Marque **CONFERIR_DEPENDENCIAS**. Veja o que está presente.
Se houver diferenças, marque também **RESOLVER_INSTALACAO**: o pip calcula um plano sem instalar.
As seis versões vêm da execução anterior do autor; dependências transitivas podem mudar.

In [ ]:
CONFERIR_DEPENDENCIAS = False # @param {type:"boolean"}
RESOLVER_INSTALACAO = False # @param {type:"boolean"}
if CONFERIR_DEPENDENCIAS:
    if not MONTAR_DRIVE or not FROZEN.is_dir():
        raise ValueError('Conclua a etapa 3 antes de continuar.')
    from community import VERSIONS, package_status
    for package, current in package_status().items():
        print(package, 'necessário:', VERSIONS[package], '| presente:', current)
    if RESOLVER_INSTALACAO:
        plan_file = Path('/content/radar-classificador-pip-plan.json')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--dry-run', '--report',
                        str(plan_file), *[f'{k}=={v}' for k, v in VERSIONS.items()]], check=True)
        for item in json.loads(plan_file.read_text()).get('install', []):
            print('Plano:', item['metadata']['name'], item['metadata']['version'])
        print('Plano pronto. Nada foi instalado.')
else:
    print('Marque CONFERIR_DEPENDENCIAS para verificar o ambiente.')


## 5. Instale somente se necessário

Se as versões já coincidem, deixe a caixa desmarcada e execute para confirmar.
Para instalar o plano visto acima, marque **INSTALAR_DEPENDENCIAS**.
Depois da instalação, se o Colab pedir reinício, reinicie e repita Configuração e etapas 1–4.
Não use `import torch` antes da Unsloth. A instalação consome tempo e rede.

In [ ]:
INSTALAR_DEPENDENCIAS = False # @param {type:"boolean"}
AMBIENTE_OK = False
if CONFERIR_DEPENDENCIAS:
    from community import VERSIONS, require_packages, save_environment
    if INSTALAR_DEPENDENCIAS:
        if not RESOLVER_INSTALACAO:
            raise ValueError('Resolva e revise o plano da etapa 4 antes de instalar.')
        subprocess.run([sys.executable, '-m', 'pip', 'install',
                        *[f'{k}=={v}' for k, v in VERSIONS.items()]], check=True)
    require_packages()
    print('Dependências: OK. Lock privado:', save_environment(ROOT))
    check = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True)
    if check.returncode:
        raise ValueError('Conflitos no ambiente; não treine antes de resolver: ' + check.stdout)
    AMBIENTE_OK = True
    print('pip check: OK. Nenhum treinamento foi iniciado.')
else:
    print('Conclua a etapa 4 primeiro.')


## 6. Abra a interface e teste

Marque **ABRIR_INTERFACE**. O modelo é carregado uma vez por sessão.
Digite uma mensagem fictícia e clique em **Classificar**. Você verá categoria, urgência e tempo.
A interface não cria servidor nem link público e não grava histórico no Drive.
Saídas do notebook podem ser salvas pelo Colab: limpe-as antes de compartilhar sua cópia.

In [ ]:
ABRIR_INTERFACE = False # @param {type:"boolean"}
if ABRIR_INTERFACE:
    if not MONTAR_DRIVE or not globals().get('AMBIENTE_OK', False):
        raise ValueError('Conclua as etapas de código, Drive e dependências primeiro.')
    from community import require_packages
    require_packages()
    requested_identity = (str(RUN_DIR.resolve()), hashlib.sha256((RUN_DIR / 'run.json').read_bytes()).hexdigest())
    if 'panel' in globals() and globals().get('panel_identity') != requested_identity:
        raise ValueError('Outra execução já está carregada. Reinicie a sessão para trocar de adaptador.')
    if 'panel' not in globals():
        from personal_ui import launch
        panel = launch(RUN_DIR, CORPUS, FROZEN)
        panel_identity = requested_identity
    else:
        print('Interface já carregada nesta sessão. Use o campo de mensagem acima.')
else:
    print('Interface desligada. Marque ABRIR_INTERFACE para testar seu adaptador.')


## Ao terminar

Confira no Drive `run.json`, `adapter/` e `comparison.json`. Não compartilhe a pasta do Drive.
Para retomar, use `TESTAR-ADAPTADOR.ipynb` com os mesmos nomes de pasta e execução.
Encerre o runtime pelo menu do Colab para liberar a GPU; salvar o notebook não salva a VM.
Uma instalação nova pode ser necessária na próxima sessão. Não retreine com os erros do teste congelado.

O modelo pode superpriorizar cancelamentos sem prazo iminente. A qualidade real depende de
dados permitidos e rótulos independentes; veja `docs/INDEPENDENT-EVAL.md`.
